# Semiconductor Pass/Fail Prediction with the UCI SECOM Dataset

## 1. Project Overview

This project applies a reproducible machine learning workflow to the UCI SECOM semiconductor pass/fail dataset. The focus is imbalanced classification, Random Forest experiment comparison, validation-based threshold selection, local MLflow tracking, and final holdout evaluation.

The script-based workflow is the source of truth for model experiments. This notebook is the portfolio summary: it reads the generated metrics and figures, adds lightweight EDA, and explains the final results in a clear interview format.


## 2. Problem Framing

The modeling goal is to identify patterns that can help flag likely fail cases under class imbalance. In this dataset, fail cases are rare, so raw accuracy is not the right main story.

A lower threshold can catch more fail cases, but it also increases the number of samples flagged by the model. The useful trade-off is therefore between fail-class recall and the flagged sample rate.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "secom.data").exists() and (candidate / "outputs" / "metrics").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root. Run this notebook from the repository root or notebooks folder.")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
METRICS_DIR = REPO_ROOT / "outputs" / "metrics"
FIGURES_DIR = REPO_ROOT / "outputs" / "figures"

required_files = [
    Path("data/secom.data"),
    Path("data/secom_labels.data"),
    Path("outputs/metrics/validation_metrics.csv"),
    Path("outputs/metrics/rf_improvement_table.csv"),
    Path("outputs/metrics/final_test_metrics.csv"),
    Path("outputs/metrics/final_feature_importance.csv"),
    Path("outputs/figures/final_roc_curve.png"),
    Path("outputs/figures/final_pr_curve.png"),
]
missing_files = [str(path) for path in required_files if not (REPO_ROOT / path).exists()]
if missing_files:
    raise FileNotFoundError("Required project files are missing: " + ", ".join(missing_files))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print("Repository paths configured.")


## 3. Dataset Summary

The workflow uses the public UCI SECOM files stored in `data/`. Labels are mapped as `-1 -> 0` for pass and `1 -> 1` for fail.


In [ ]:
# Load SECOM data for lightweight EDA only.
X_raw = pd.read_csv(DATA_DIR / "secom.data", sep=r"\s+", header=None, engine="python")
X_raw.columns = [f"sensor_{index:03d}" for index in range(1, X_raw.shape[1] + 1)]

labels_df = pd.read_csv(DATA_DIR / "secom_labels.data", sep=r"\s+", header=None, engine="python")
labels_df.columns = ["label", "date_part", "time_part"]
labels_df["timestamp"] = pd.to_datetime(
    labels_df["date_part"].astype(str).str.replace('"', "", regex=False)
    + " "
    + labels_df["time_part"].astype(str).str.replace('"', "", regex=False),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)

y = labels_df["label"].replace({-1: 0, 1: 1}).astype(int)
y.name = "target"

class_counts = y.value_counts().sort_index()
pass_count = int(class_counts.get(0, 0))
fail_count = int(class_counts.get(1, 0))
fail_rate = fail_count / len(y)

dataset_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "loaded_sensor_features",
            "pass_samples",
            "fail_samples",
            "fail_rate_pct",
            "first_timestamp",
            "last_timestamp",
        ],
        "value": [
            X_raw.shape[0],
            X_raw.shape[1],
            pass_count,
            fail_count,
            round(100 * fail_rate, 2),
            labels_df["timestamp"].min(),
            labels_df["timestamp"].max(),
        ],
    }
)

display(dataset_summary)


## 4. EDA and Data Quality Checks

The dataset is small, high-dimensional, and imbalanced. The checks below summarize class balance, missing values, low-information columns, and simple timestamp coverage.


In [ ]:
missing_ratio = X_raw.isna().mean()
missing_count = X_raw.isna().sum()
unique_counts = X_raw.nunique(dropna=True)
constant_features = int((unique_counts <= 1).sum())
near_constant_features = int((unique_counts <= 2).sum())

eda_summary = pd.DataFrame(
    {
        "metric": [
            "features_with_any_missing",
            "features_above_30pct_missing",
            "features_above_50pct_missing",
            "constant_features",
            "near_constant_features",
        ],
        "value": [
            int((missing_ratio > 0).sum()),
            int((missing_ratio > 0.30).sum()),
            int((missing_ratio > 0.50).sum()),
            constant_features,
            near_constant_features,
        ],
    }
)

top_missing_features = missing_ratio.sort_values(ascending=False).head(10).index
top_missing = pd.DataFrame(
    {
        "feature": top_missing_features,
        "missing_ratio": missing_ratio.loc[top_missing_features].values,
        "missing_count": missing_count.loc[top_missing_features].values,
    }
)

display(eda_summary)
display(top_missing)


In [ ]:
# Lightweight EDA plots; no model training happens here.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

class_counts.plot(kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Target Class Distribution")
axes[0].set_xticklabels(["Pass (0)", "Fail (1)"], rotation=0)
axes[0].set_ylabel("Count")

axes[1].hist(missing_ratio, bins=30, color="#72B7B2", edgecolor="white")
axes[1].set_title("Missing Ratio by Sensor")
axes[1].set_xlabel("Missing ratio")
axes[1].set_ylabel("Feature count")

weekly_fail_rate = (
    pd.DataFrame({"timestamp": labels_df["timestamp"], "target": y})
    .dropna()
    .set_index("timestamp")
    .resample("W")["target"]
    .mean()
)
weekly_fail_rate.plot(ax=axes[2], color="#F58518")
axes[2].set_title("Weekly Fail Rate")
axes[2].set_xlabel("Week")
axes[2].set_ylabel("Fail rate")

plt.tight_layout()
plt.show()


In [ ]:
observations = [
    f"- The fail class is rare: {fail_count} of {len(y)} samples are labeled as fail.",
    f"- The feature matrix has {X_raw.shape[1]} loaded sensor columns, which is high-dimensional relative to {fail_count} fail cases.",
    f"- {int((missing_ratio > 0).sum())} sensor columns contain at least one missing value.",
    f"- {constant_features} constant features and {near_constant_features} near-constant features support the need for preprocessing.",
    "- The timestamps make time-based validation a reasonable future improvement, but this project uses a stratified split for the current workflow.",
]

display(Markdown("### EDA Observations\n" + "\n".join(observations)))


## 5. Why Accuracy Is Misleading

The fail class is rare. A model can have high raw accuracy while still missing the fail cases that are most important for screening.

The table below reads validation results from `outputs/metrics/validation_metrics.csv` and compares a majority baseline with two Random Forest operating points.


In [ ]:
validation_metrics = pd.read_csv(METRICS_DIR / "validation_metrics.csv")
rf_improvement = pd.read_csv(METRICS_DIR / "rf_improvement_table.csv")
final_test_metrics = pd.read_csv(METRICS_DIR / "final_test_metrics.csv")
final_feature_importance = pd.read_csv(METRICS_DIR / "final_feature_importance.csv")

selected_experiment_name = str(final_test_metrics.iloc[0]["selected_experiment_name"])
selected_validation_rows = validation_metrics.loc[
    validation_metrics["experiment_name"] == selected_experiment_name
]
if selected_validation_rows.empty:
    raise ValueError(f"Selected experiment is missing from validation metrics: {selected_experiment_name}")
selected_validation = selected_validation_rows.iloc[0]

comparison_names = [
    "dummy_majority_baseline",
    "rf_current_config_threshold_050",
    selected_experiment_name,
]
accuracy_example = validation_metrics[
    validation_metrics["experiment_name"].isin(dict.fromkeys(comparison_names))
][
    [
        "experiment_name",
        "threshold",
        "accuracy",
        "recall",
        "f2",
        "balanced_accuracy",
        "tp",
        "fp",
        "fn",
        "tn",
        "review_count",
        "review_rate",
    ]
].rename(
    columns={
        "review_count": "flagged_sample_count",
        "review_rate": "flagged_sample_rate",
    }
)

display(accuracy_example)

current_050 = validation_metrics.loc[
    validation_metrics["experiment_name"] == "rf_current_config_threshold_050"
].iloc[0]
validation_fail_total = int(selected_validation["tp"] + selected_validation["fn"])
validation_total = int(selected_validation[["tp", "fp", "fn", "tn"]].sum())

summary_text = (
    f"At threshold {current_050['threshold']:.2f}, the current Random Forest configuration detected "
    f"{int(current_050['tp'])} of {validation_fail_total} validation fail cases. "
    f"With the validation-selected threshold {selected_validation['threshold']:.3f}, it detected "
    f"{int(selected_validation['tp'])} of {validation_fail_total} validation fail cases and flagged "
    f"{int(selected_validation['review_count'])} of {validation_total} validation samples."
)
display(Markdown(summary_text))


## 6. Reproducible Workflow and Leakage Control

The script workflow uses a stratified train/validation/test split:

- training data fits preprocessing and model parameters
- validation data compares experiments and selects thresholds
- test data is held back for one final evaluation

Missingness filtering and imputation are fit on training data only. The notebook does not select models or tune thresholds.


## 7. Baseline and Random Forest Experiment Design

The validation experiments include:

- majority-class dummy baseline
- logistic regression with PCA baseline
- default Random Forest at threshold `0.50`
- class-balanced Random Forest at threshold `0.50`
- current Random Forest configuration at threshold `0.50`
- current Random Forest configuration with a validation-selected threshold
- small RandomizedSearchCV Random Forest comparison at threshold `0.50`
- the same randomized-search model with a validation-selected threshold

The randomized search is a small comparison step, not a broad production tuning process.


## 8. Random Forest Iterative Results

The table below is read from `outputs/metrics/rf_improvement_table.csv`. It focuses on fail-class recall, F2-score, balanced accuracy, PR-AUC, confusion counts, and flagged sample rate.


In [ ]:
rf_columns = [
    "experiment_name",
    "threshold",
    "recall",
    "f2",
    "balanced_accuracy",
    "pr_auc",
    "tp",
    "fp",
    "fn",
    "tn",
    "review_count",
    "review_rate",
]
rf_table = rf_improvement[rf_columns].rename(
    columns={
        "review_count": "flagged_sample_count",
        "review_rate": "flagged_sample_rate",
    }
)
display(rf_table)

display(
    Markdown(
        f"The final validation candidate is `{selected_experiment_name}`. Its threshold came from validation data, "
        f"and it flagged {int(selected_validation['review_count'])} validation samples."
    )
)


## 9. Threshold Selection and Flagged Sample Rate

The script workflow selects tuned thresholds on validation data using F2-score. F2-score weights recall more than precision.

The source CSV uses the metric name `review_rate`. In this notebook, `review_rate` is interpreted as the flagged sample rate:

`review_rate = (TP + FP) / total_samples`

A lower threshold can improve fail detection, but it also increases false positives and the flagged sample rate.


In [ ]:
selected_validation_table = selected_validation_rows[
    [
        "experiment_name",
        "threshold_source",
        "threshold",
        "recall",
        "f2",
        "balanced_accuracy",
        "precision",
        "tp",
        "fp",
        "fn",
        "tn",
        "review_count",
        "review_rate",
    ]
].rename(
    columns={
        "review_count": "flagged_sample_count",
        "review_rate": "flagged_sample_rate",
    }
)

display(selected_validation_table)


## 10. MLflow Experiment Tracking Summary

The script workflow uses local MLflow tracking:

- tracking URI: `sqlite:///mlflow.db`
- experiment name: `secom-pass-fail-screening`
- runs log model settings, validation metrics, selected thresholds, confusion counts, and artifacts
- local MLflow files are ignored by Git

The notebook does not require the MLflow UI to be running.


## 11. Final Holdout Test Evaluation

The selected validation candidate was evaluated once on the untouched holdout test set. The table and summary below are read from `outputs/metrics/final_test_metrics.csv`.


In [ ]:
final_columns = [
    "selected_experiment_name",
    "threshold",
    "recall",
    "f2",
    "balanced_accuracy",
    "pr_auc",
    "roc_auc",
    "tp",
    "fp",
    "fn",
    "tn",
    "review_count",
    "review_rate",
]
final_table = final_test_metrics[final_columns].rename(
    columns={
        "review_count": "flagged_sample_count",
        "review_rate": "flagged_sample_rate",
    }
)
display(final_table)

final_row = final_test_metrics.iloc[0]
test_fail_total = int(final_row["tp"] + final_row["fn"])
test_total = int(final_row[["tp", "fp", "fn", "tn"]].sum())
final_summary = (
    f"The final model `{final_row['selected_experiment_name']}` used threshold {final_row['threshold']:.3f}. "
    f"On the holdout test split, it detected {int(final_row['tp'])} of {test_fail_total} fail cases and "
    f"flagged {int(final_row['review_count'])} of {test_total} samples. "
    f"The flagged sample rate was {final_row['review_rate']:.4f}."
)
display(Markdown(final_summary))


### Final Confusion Matrix

The confusion matrix below is generated directly from `final_test_metrics.csv`.


In [ ]:
tn = int(final_row["tn"])
fp = int(final_row["fp"])
fn = int(final_row["fn"])
tp = int(final_row["tp"])
confusion_matrix = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion_matrix, cmap="Blues")
ax.set_xticks([0, 1], labels=["Predicted pass", "Predicted fail"])
ax.set_yticks([0, 1], labels=["Actual pass", "Actual fail"])
ax.set_title("Final Holdout Confusion Matrix")

for row in range(confusion_matrix.shape[0]):
    for col in range(confusion_matrix.shape[1]):
        ax.text(col, row, confusion_matrix[row, col], ha="center", va="center", color="black")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


### Final ROC and PR Curves

These curve figures were generated by `scripts/evaluate_final_model.py` during final evaluation.

![Final ROC curve](../outputs/figures/final_roc_curve.png)

![Final PR curve](../outputs/figures/final_pr_curve.png)


## 12. Feature Importance and Interpretation

Feature importance is treated as model-driven signal ranking, not process-causal explanation. The SECOM sensors are anonymous, so these rankings should not be interpreted as physical root causes.


In [ ]:
top_features = final_feature_importance.head(20).copy()
display(top_features)

fig, ax = plt.subplots(figsize=(8, 6))
plot_data = top_features.iloc[::-1]
ax.barh(plot_data["feature"], plot_data["importance"], color="#4C78A8")
ax.set_title("Top Final Random Forest Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 13. Limitations

Main limitations:

- the fail class is small
- one stratified split is used
- the validation and test splits each contain a small number of fail cases
- time-based validation is not implemented yet
- there is no real engineering cost function for false positives and false negatives
- sensor names are anonymous
- feature importance does not explain physical cause
- the project is a portfolio workflow, not a deployed system


## 14. Final Takeaways

The main takeaway is that threshold choice matters for fail-class screening.

Raw accuracy is not the right story for this dataset. The more useful view is recall, F2-score, balanced accuracy, PR-AUC, confusion counts, and flagged sample rate.

The reusable scripts, MLflow tracking, generated CSV outputs, and final notebook make the project easier to reproduce and explain.


## 15. How to Reproduce the Script Workflow

Run these commands from the repository root:

```bash
python -m pytest
python -m ruff check .
python scripts/run_rf_experiments.py --config configs/rf_experiments.yaml
python scripts/evaluate_final_model.py --config configs/final_rf.yaml
python scripts/export_experiment_summary.py
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

The notebook can be run after the metrics and figure outputs have been generated by the scripts.
